# Datenaufbereitung: ergänzende Datensätze

Liest einen ergänzenden Datensatz aus `data/additional/` ein (ein Land pro Zeile, ein Jahr pro Spalte), wandelt Angaben wie `12k` oder `3.4M` in Zahlen um und speichert die Werte in der Tabelle `metric`.

Für jeden Datensatz werden `dataset_file` und `metric_definition_id` angepasst.

In [132]:
import numpy as np
import pandas as pd
import re

from bina_db import connect, CountryLookup

pd.set_option('display.max_rows', 250)

In [233]:
# Datensatz und zugehörige Metrikdefinition (ID in der Tabelle «metric_definition»)
dataset_file = "../data/additional/sustainable_development_index.csv"
metric_definition_id = 20

data = pd.read_csv(dataset_file, delimiter=";", encoding="utf-8-sig")

In [231]:
# Funktion zur Umwandlung von Werten
def convert_k(value):
    if isinstance(value, str):  # Prüfe, ob der Wert ein String ist
        # Einige Quellen verwenden das Unicode-Minus (U+2212) statt '-', das float() nicht versteht
        value = value.replace("\u2212", "-")

        match_kilo = re.search(r'^(\d+(?:\.\d+)?)(k)$', value, re.IGNORECASE)
        if match_kilo:
            # Umwandlung des numerischen Teils und Multiplikation mit 1000
            num_part = float(match_kilo.group(1))
            return int(num_part * 1000)

        match_million = re.search(r'^(\d+(?:\.\d+)?)(M)$', value, re.IGNORECASE)
        if match_million:
            # Umwandlung des numerischen Teils und Multiplikation mit 1000000
            num_part = float(match_million.group(1))
            return int(num_part * 1000000)

        match_billion = re.search(r'^(\d+(?:\.\d+)?)(B)$', value, re.IGNORECASE)
        if match_billion:
            # Umwandlung des numerischen Teils und Multiplikation mit 1000000000
            num_part = float(match_billion.group(1))
            return int(num_part * 1000000000)
    return value

data = data.map(convert_k)

In [234]:
data

,country,1990,1991,1992,1993,1994,1995,1996,1997,1998,...,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019
0,Afghanistan,32.5,33.1,34.0,33.5,33.0,35.6,36.0,36.6,37.1,...,50.8,51.4,52.7,53.4,53.8,53.8,54.1,54.5,54.9,55.1
1,Angola,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,55.8,57.4,58.6,59.8,60.8,61.6,62.2,62.7,62.7,62.6
2,Albania,69.9,68.0,66.2,66.5,67.2,68.6,69.5,69.5,70.6,...,79.6,81.1,82.2,82.6,82.6,82.3,82.1,82.1,82.3,82.6
3,UAE,46.9,29.2,19.4,12.8,10.6,11.1,9.9,10.2,10.7,...,14.7,13.6,13.2,11.9,11.5,11.0,10.8,10.9,11.0,11.0
4,Argentina,77.1,77.1,77.1,77.5,77.4,77.8,78.0,78.3,78.0,...,81.6,80.6,80.3,78.7,78.2,77.6,77.2,77.2,77.2,77.7
5,Armenia,NaN,69.9,66.5,66.4,66.8,67.5,68.3,69.5,70.7,...,80.4,80.8,81.3,81.8,82.0,82.1,81.7,82.0,82.2,82.7
6,Antigua and Barbuda,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,65.9,63.0,64.9,63.5,62.1,62.0,61.7,61.6,61.9,62.2
7,Australia,33.6,32.7,35.1,35.4,33.7,30.1,28.5,27.3,24.3,...,18.0,17.2,16.7,16.4,15.8,15.3,15.1,15.0,15.0,15.0
8,Austria,68.0,64.4,60.3,61.7,58.0,55.3,53.8,53.8,49.3,...,30.2,28.5,28.1,27.2,26.9,26.0,25.5,25.1,25.2,25.2
9,Azerbaijan,NaN,NaN,NaN,NaN,NaN,64.5,64.8,65.4,66.5,...,77.6,77.4,77.6,77.8,78.1,78.3,79.1,79.4,79.4,79.6


In [ ]:
maria_db_connection = connect()
country_lookup = CountryLookup(maria_db_connection)

insert_statement = "insert into metric(value, year, country, metric_definition) VALUES (?, ?, ?, ?)"

cursor = maria_db_connection.cursor()

for yearIdentifier in data.columns[1:]:
    for index, row in data[["country", str(yearIdentifier)]].iterrows():
        country_name = row["country"]
        yearData = row[str(yearIdentifier)]

        if not np.isnan(float(yearData)):
            country_id = country_lookup.lookup(country_name)

            if country_id is None:
                print("Could not resolve country: " + country_name)
                continue

            cursor.execute(insert_statement, (yearData,
                                              yearIdentifier,
                                              country_id,
                                              metric_definition_id
                                              ))
            maria_db_connection.commit()

maria_db_connection.close()